## 07. Modeliranje: Eksperiment C (MC + classical FE + ML)

U ovom notebook-u treniramo klasične ML modele (multi-label klasifikacija) nad Multi-Class (MC) podskupom proteina. Koristimo feature-set kombinacije definisane u notebook-u 03: AAC-CV, TF-IDF + SVD i PH.

### Uvoz biblioteka

In [2]:
import os
import numpy as np
import pandas as pd
import joblib

from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    f1_score, hamming_loss, accuracy_score,
    classification_report, make_scorer
)

from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

### Učitavanje podataka

Koristimo isti MC train/test split i isti MultiLabelBinarizet koji su definisani u notebook-u 03 (nema ponovnog fitovanja).

In [21]:
DATA_DIR = "../data/processed"
FEATURES_DIR = "../data/features"
RESULTS_DIR = "../data/results"
MODELS_DIR = "../data/models"

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

RANDOM_STATE = 42

# Entry liste za MC train/test skup (03 notebook)
entries_mc_train = pd.read_csv(os.path.join(FEATURES_DIR, "mc_train_entries.csv"))["Entry"].values
entries_mc_test = pd.read_csv(os.path.join(FEATURES_DIR, "mc_test_entries.csv"))["Entry"].values

# Multi-hot labele (fitovane u 03 - MultiLabelBinarizer)
mlb = joblib.load(os.path.join(FEATURES_DIR, "mc_label_binarizer.pkl"))
label_columns = list(mlb.classes_)

# Labels
mc_labels_df = pd.read_csv(os.path.join(FEATURES_DIR, "mc_labels.csv"), index_col="Entry")
mc_labels_df

y_mc_train = mc_labels_df.loc[entries_mc_train, label_columns].values
y_mc_test = mc_labels_df.loc[entries_mc_test, label_columns].values

print(f"Train skup: {len(entries_mc_train)} proteina")
print(f"Test skup: {len(entries_mc_test)} proteina")
print(f"Klase: {label_columns}")
print(f"y_mc_train: {y_mc_train.shape} | y_mc_test: {y_mc_test.shape}")

Train skup: 5643 proteina
Test skup: 1400 proteina
Klase: ['Hydrolase', 'Receptor', 'Structural protein', 'Transcription factor', 'Transport protein']
y_mc_train: (5643, 5) | y_mc_test: (1400, 5)


### AAC-CV

In [23]:
# AAC-CV je izračunat jednom za cijeli dataset (fiksni vokabular, ne zahtijeva fitovanje)
# Selektujemo redove koji pripadaju MC train/test skupu
aac_df = pd.read_csv(os.path.join(FEATURES_DIR, "aac_cv_features.csv"), index_col="Entry")

mc_train_aac = aac_df.loc[entries_mc_train].values
mc_test_aac = aac_df.loc[entries_mc_test].values

print(f"AAC-CV train: {mc_train_aac.shape} | test: {mc_test_aac.shape}")

AAC-CV train: (5643, 20) | test: (1400, 20)


### TF-IDF + SVD

In [25]:
# Fitovano na MC train skupu u 03
mc_train_tfidf = np.load(os.path.join(FEATURES_DIR, "mc_train_tfidf_svd.npy")) 
mc_test_tfidf = np.load(os.path.join(FEATURES_DIR, "mc_test_tfidf_svd.npy")) 

print(f"TF-IDF-SVD train: {mc_train_tfidf.shape} | test: {mc_test_tfidf.shape}")

TF-IDF-SVD train: (5643, 150) | test: (1400, 150)


### Fizičko-hemijske osobine

In [26]:
mc_train_ph = np.load(os.path.join(FEATURES_DIR, "mc_train_physchem_scaled.npy"))
mc_test_ph = np.load(os.path.join(FEATURES_DIR, "mc_test_physchem_scaled.npy"))

print(f"PH train: {mc_train_ph.shape} | test: {mc_test_ph.shape}")

PH train: (5643, 5) | test: (1400, 5)


### Kombinovanje u feature setove

In [27]:
feature_sets_train = {
    "AAC-CV": mc_train_aac,
    "TFIDF-SVD": mc_train_tfidf,
    "PH": mc_train_ph,
    "AAC-CV+TFIDF-SVD": np.hstack([mc_train_aac, mc_train_tfidf]),
    "AAC-CV+PH": np.hstack([mc_train_aac, mc_train_ph]),
    "TFIDF-SVD+PH": np.hstack([mc_train_tfidf, mc_train_ph]),
}

feature_sets_test = {
    "AAC-CV": mc_test_aac,
    "TFIDF-SVD": mc_test_tfidf,
    "PH": mc_test_ph,
    "AAC-CV+TFIDF-SVD": np.hstack([mc_test_aac, mc_test_tfidf]),
    "AAC-CV+PH": np.hstack([mc_test_aac, mc_test_ph]),
    "TFIDF-SVD+PH": np.hstack([mc_test_tfidf, mc_test_ph]),
}

for name, matrix in feature_sets_train.items():
    print(f"{name:<20} train: {matrix.shape} test: {feature_sets_test[name].shape}")

AAC-CV               train: (5643, 20) test: (1400, 20)
TFIDF-SVD            train: (5643, 150) test: (1400, 150)
PH                   train: (5643, 5) test: (1400, 5)
AAC-CV+TFIDF-SVD     train: (5643, 170) test: (1400, 170)
AAC-CV+PH            train: (5643, 25) test: (1400, 25)
TFIDF-SVD+PH         train: (5643, 155) test: (1400, 155)


### Definisanje modela i mreže hiperparametara

In [31]:
param_grids = {
    "LogisticRegression": {
        "model": OneVsRestClassifier(
            LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)
        ),
        "params": {
            "estimator__C": [0.01, 0.1, 1, 10, 100],
        },
    },
    "RandomForest": {
        "model": OneVsRestClassifier(
            RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
        ),
        "params": {
            "estimator__n_estimators": [200, 400],
            "estimator__max_depth": [10, 20, None],
        },
    },
    "SVM": {
        "model": OneVsRestClassifier(
            SVC(class_weight="balanced", probability=True, random_state=RANDOM_STATE)
        ),
        "params": {
            "estimator__C": [0.1, 1, 10],
            "estimator__kernel": ["rbf", "linear"],
        },
    },
}

f1_micro_scorer = make_scorer(f1_score, average="micro", zero_division=0)
cv = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

### Glavna petlja treniranja

In [32]:
all_results = []

for feature_name, X_train in feature_sets_train.items():
    X_test = feature_sets_test[feature_name]

    for model_name, config in param_grids.items():
        print(f"Treniranje: {model_name:<20} | Feature set: {feature_name}")

        grid = GridSearchCV(
            estimator=config["model"],
            param_grid=config["params"],
            cv=cv,
            scoring=f1_micro_scorer,
            n_jobs=-1,
            refit=True,
        )
        grid.fit(X_train, y_mc_train)

        best_model = grid.best_estimator_

        # Predikcije na train skupu (isti fit-ovan model, bez dodatnog treniranja)
        y_train_pred = best_model.predict(X_train)
        train_f1_micro = f1_score(y_mc_train, y_train_pred, average="micro", zero_division=0)
        train_f1_macro = f1_score(y_mc_train, y_train_pred, average="macro", zero_division=0)
        train_acc = accuracy_score(y_mc_train, y_train_pred)  # subset accuracy (exact match)

        # Predikcije na test skupu
        y_test_pred = best_model.predict(X_test)
        test_f1_micro = f1_score(y_mc_test, y_test_pred, average="micro", zero_division=0)
        test_f1_macro = f1_score(y_mc_test, y_test_pred, average="macro", zero_division=0)
        test_f1_samples = f1_score(y_mc_test, y_test_pred, average="samples", zero_division=0)
        test_acc = accuracy_score(y_mc_test, y_test_pred)  # subset accuracy (exact match)
        test_hamming = hamming_loss(y_mc_test, y_test_pred)

        all_results.append({
            "Feature set": feature_name,
            "Model": model_name,
            "Best params": grid.best_params_,

            "CV F1 micro": grid.best_score_,

            "Train F1 micro": train_f1_micro,
            "Train F1 macro": train_f1_macro,
            "Train Subset Accuracy": train_acc,

            "Test F1 micro": test_f1_micro,
            "Test F1 macro": test_f1_macro,
            "Test F1 samples": test_f1_samples,
            "Test Subset Accuracy": test_acc,
            "Test Hamming Loss": test_hamming,
            
            "Overfit Gap (F1 micro)": train_f1_micro - test_f1_micro,
        })

        model_filename = f"mc_{model_name}_{feature_name}.pkl"
        joblib.dump(best_model, os.path.join(MODELS_DIR, model_filename))

print("\nTreniranje završeno.")

Treniranje: LogisticRegression   | Feature set: AAC-CV
Treniranje: RandomForest         | Feature set: AAC-CV
Treniranje: SVM                  | Feature set: AAC-CV
Treniranje: LogisticRegression   | Feature set: TFIDF-SVD
Treniranje: RandomForest         | Feature set: TFIDF-SVD
Treniranje: SVM                  | Feature set: TFIDF-SVD
Treniranje: LogisticRegression   | Feature set: PH
Treniranje: RandomForest         | Feature set: PH
Treniranje: SVM                  | Feature set: PH
Treniranje: LogisticRegression   | Feature set: AAC-CV+TFIDF-SVD
Treniranje: RandomForest         | Feature set: AAC-CV+TFIDF-SVD
Treniranje: SVM                  | Feature set: AAC-CV+TFIDF-SVD
Treniranje: LogisticRegression   | Feature set: AAC-CV+PH
Treniranje: RandomForest         | Feature set: AAC-CV+PH
Treniranje: SVM                  | Feature set: AAC-CV+PH
Treniranje: LogisticRegression   | Feature set: TFIDF-SVD+PH
Treniranje: RandomForest         | Feature set: TFIDF-SVD+PH
Treniranje: SVM  

### Pregled i čuvanje rezultata

In [ ]:
results_df = pd.DataFrame(all_results)

# Sortiranje po Test F1 micro
results_df = results_df.sort_values(by="Test F1 micro", ascending=False).reset_index(drop=True)
results_df.to_csv(os.path.join(RESULTS_DIR, "mc_classical_modeling_results.csv"), index=False)

display_cols = [
    "Model", "Feature set",
    "Train F1 micro", "Train Subset Accuracy",
    "CV F1 micro",
    "Test F1 micro", "Test F1 macro", "Test F1 samples",
    "Test Subset Accuracy", "Test Hamming Loss",
    "Overfit Gap (F1 micro)",
]

results_df[display_cols].style.format({
    "Train F1 micro": "{:.3f}",
    "Train Subset Accuracy": "{:.3f}",
    "CV F1 micro": "{:.3f}",
    "Test F1 micro": "{:.3f}",
    "Test F1 macro": "{:.3f}",
    "Test F1 samples": "{:.3f}",
    "Test Subset Accuracy": "{:.3f}",
    "Test Hamming Loss": "{:.3f}",
    "Overfit Gap (F1 micro)": "{:.3f}",
}).background_gradient(subset=["Overfit Gap (F1 micro)"], cmap="Reds")

,Model,Feature set,Train F1 micro,Train Subset Accuracy,CV F1 micro,Test F1 micro,Test F1 macro,Test F1 samples,Test Subset Accuracy,Test Hamming Loss,Overfit Gap (F1 micro)
0,SVM,AAC-CV+TFIDF-SVD,0.982,0.963,0.810,0.828,0.818,0.820,0.741,0.074,0.155
1,SVM,TFIDF-SVD,0.987,0.973,0.812,0.825,0.813,0.810,0.743,0.074,0.162
2,SVM,TFIDF-SVD+PH,0.879,0.770,0.782,0.777,0.762,0.789,0.654,0.102,0.101
3,RandomForest,AAC-CV+TFIDF-SVD,0.945,0.886,0.724,0.740,0.713,0.697,0.640,0.105,0.205
4,RandomForest,TFIDF-SVD+PH,0.937,0.872,0.733,0.738,0.713,0.707,0.636,0.108,0.198
5,RandomForest,AAC-CV+PH,0.918,0.834,0.732,0.736,0.720,0.711,0.624,0.112,0.182
6,SVM,AAC-CV,0.809,0.649,0.727,0.732,0.721,0.754,0.569,0.129,0.077
7,RandomForest,AAC-CV,0.914,0.826,0.715,0.728,0.707,0.692,0.612,0.113,0.186
8,RandomForest,TFIDF-SVD,0.942,0.885,0.711,0.710,0.681,0.662,0.606,0.116,0.233
9,LogisticRegression,TFIDF-SVD+PH,0.723,0.501,0.700,0.705,0.693,0.726,0.481,0.149,0.018


### Classification report za najbolji model

In [34]:
best_row = results_df.iloc[0]
print(f"Najbolja kombinacija: {best_row['Model']} + {best_row['Feature set']}")
print(f"Parametri: {best_row['Best params']}\n")

best_model_path = os.path.join(MODELS_DIR, f"mc_{best_row['Model']}_{best_row['Feature set']}.pkl")
best_model = joblib.load(best_model_path)
best_X_test = feature_sets_test[best_row["Feature set"]]

y_pred_best = best_model.predict(best_X_test)
print(classification_report(y_mc_test, y_pred_best, target_names=label_columns, zero_division=0))

Najbolja kombinacija: SVM + AAC-CV+TFIDF-SVD
Parametri: {'estimator__C': 10, 'estimator__kernel': 'rbf'}

                      precision    recall  f1-score   support

           Hydrolase       0.76      0.91      0.83       481
            Receptor       0.80      0.83      0.81       320
  Structural protein       0.95      0.61      0.74       154
Transcription factor       0.94      0.92      0.93       283
   Transport protein       0.79      0.76      0.77       248

           micro avg       0.82      0.84      0.83      1486
           macro avg       0.85      0.81      0.82      1486
        weighted avg       0.83      0.84      0.83      1486
         samples avg       0.81      0.85      0.82      1486

